# Sentry Loop — three-agent GRPO training

Use a CUDA GPU runtime for language-model training. This notebook runs the current `train.py`, which logs real trainer metrics through Trackio. The Arena uses the original demo policies; this notebook trains language-model weights separately.

The locked dependency/configuration preflight has passed. A new full Qwen GPU training run has not been verified in the September 13 demo. The original Unsloth notebook is preserved as `legacy_colab_training.ipynb` for reference.

## 1. Upload the current source checkout

The upstream default branch does not contain today's changes. In your local checkout, create a credential-free source archive:

```bash
uv run python -c "from scripts.deploy_tenki import source_bundle; from pathlib import Path; Path('.sentry/colab-source.tar.gz').write_bytes(source_bundle())"
```

Upload that archive in the next cell. It excludes `.env` and all account credentials.


In [ ]:
from google.colab import files
from pathlib import Path
import tarfile

uploaded = files.upload()
source = next(name for name in uploaded if name.endswith('.tar.gz'))
checkout = Path('/content/sentry-loop')
checkout.mkdir(exist_ok=True)
with tarfile.open(source) as archive:
    archive.extractall(checkout, filter='data')
%cd /content/sentry-loop
!pip -q install uv
!uv sync --frozen --python 3.12 --extra train


## 2. Configure optional Hugging Face graph publication

Add `HF_TOKEN` as a private Colab secret and grant this notebook access. Set your own Space ID below. Static graph Spaces publish on completion; they do not require a Pro plan. Leave the Space blank to keep training metrics local. Never paste a token into notebook source or output.


In [ ]:
import os
from google.colab import userdata

TRACKIO_SPACE = os.environ.get('SENTRY_TRACKIO_SPACE', '')  # Set this to namespace/space.
if TRACKIO_SPACE:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    os.environ['SENTRY_TRACKIO_SPACE'] = TRACKIO_SPACE
    os.environ['SENTRY_TRACKIO_MODE'] = 'static'
os.environ['SENTRY_DATA_DIR'] = '/content/sentry-loop/.sentry'


## 3. Validate the installed configuration

This checks the dataset, dependency versions, and valid GRPO group sizes. It does not download weights or train. If CUDA is unavailable, select a GPU runtime before continuing.


In [ ]:
import subprocess
subprocess.run(['uv', 'run', '--extra', 'train', 'python', 'train.py', '--check_setup',
    '--num_episodes', '2', '--max_steps', '2', '--device', 'cuda'], check=True)


## 4. Train the selected role

The trainer preserves the original Transformers full-model path (or Unsloth LoRA when configured). It logs loss, rewards, and available KL/completion metrics. The original Arena demo does not load these weights automatically.


In [ ]:
TARGET_AGENT = 'worker'  # worker, attacker, or oversight
MODEL = 'Qwen/Qwen2.5-1.5B-Instruct'
subprocess.run(['uv', 'run', '--extra', 'train', 'python', 'train.py',
    '--agent', TARGET_AGENT, '--model_name', MODEL, '--device', 'cuda',
    '--max_steps', '80', '--num_episodes', '50', '--num_generations', '8'], check=True)


## 5. Inspect actual metrics and download artifacts

Select the new run in the HF graph Space after publication. An app sharing this data directory can display the CSV during training. Copy the `.sentry/training/<id>` folder to another app's data directory to inspect it there.


In [ ]:
import csv
import json
import matplotlib.pyplot as plt

runs = [p for p in Path('.sentry/training').glob('*/metadata.json')
        if json.loads(p.read_text()).get('kind') == 'grpo']
metadata_path = max(runs, key=lambda p: json.loads(p.read_text())['created_at'])
metadata = json.loads(metadata_path.read_text())
with metadata_path.with_name('metrics.csv').open() as f:
    rows = list(csv.DictReader(f))
fig, axes = plt.subplots(1, 2, figsize=(11, 3))
for axis, key in zip(axes, ['loss', 'reward']):
    values = [(int(r['step']), float(r[key])) for r in rows if r.get(key)]
    if values:
        axis.plot(*zip(*values))
    axis.set(xlabel='Optimizer step', ylabel=key, title=key.capitalize())
plt.show()
print('Run:', metadata['id'], 'Status:', metadata['status'])
print('Graph publication:', metadata.get('tracking', {}).get('status'))


In [ ]:
import tarfile
artifact_path = Path('/content/sentry-loop-training-artifacts.tar.gz')
with tarfile.open(artifact_path, 'w:gz') as archive:
    archive.add(metadata_path.parent, arcname=f'.sentry/training/{metadata_path.parent.name}')
    model_path = Path(f'sentinelops-grpo-{TARGET_AGENT}')
    if model_path.exists():
        archive.add(model_path, arcname=model_path.name)
files.download(str(artifact_path))
